In [8]:
import wrds
import pandas as pd
import os
import time     
from edgar import Filing, set_identity, find, Company
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import time

ImportError: cannot import name 'Filing' from 'edgar' (/home/rens/anaconda3/lib/python3.12/site-packages/edgar/__init__.py)

In [3]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
query = """
SELECT DISTINCT c.cik, c.conm, cr.permno,
       ABS(cr.prc) * cr.shrout AS market_cap,
       cr.prc,
       cr.vol * ABS(cr.prc) AS dollar_volume
FROM comp.company c
JOIN crsp.ccmxpf_linktable l
  ON c.gvkey = l.gvkey
JOIN crsp.dsf cr
  ON l.lpermno = cr.permno
WHERE c.cik IS NOT NULL
  AND c.fic = 'USA'
  AND cr.date = '2023-12-29'   -- pick a snapshot date
  AND l.linktype IN ('LU', 'LC')
  AND l.linkprim IN ('P', 'C')
  AND ABS(cr.prc) >= 5
  AND (ABS(cr.prc) * cr.shrout) > 2e6
  AND (cr.vol * ABS(cr.prc)) > 1e4
"""

df = db.raw_sql(query)

bad_keywords = [
    "FUND", "ETF", "TRUST", "PORTFOLIO", "SERIES",
    "INVESTMENT", "SICAV"
]

df = df[~df['conm'].str.upper().str.contains('|'.join(bad_keywords))]

print(df.head())
print(f"Total unique CIKs: {len(df)}")

          cik                          conm  permno      market_cap  \
0  0000001750                      AAR CORP   54594       2215387.2   
1  0000001800           ABBOTT LABORATORIES   20482    191088014.13   
2  0000002230   ADAMS DIVERSIFIED EQUITY FD   10065       2139545.1   
3  0000002488        ADVANCED MICRO DEVICES   61241     238214560.0   
4  0000002969  AIR PRODUCTS & CHEMICALS INC   28222  60846024.17772   

         prc   dollar_volume  
0       62.4      15618033.6  
1     110.07    390781961.28  
2      17.71      1768857.09  
3     147.41   9130728411.58  
4  273.79999  242597743.1396  
Total unique CIKs: 1461


In [ ]:
BASE_DIR = Path(__file__).resolve().parent.parent
data_dir = BASE_DIR / "data" / "RensTest"
data_dir.mkdir(parents=True, exist_ok=True)

set_identity("gerritsenREN2@gmail.com")

tickers = df['cik'].astype(str).tolist()
start_yr = 2001


MAX_WORKERS = 5  # safe

def process_ticker(ticker):
    try:
        company = Company(ticker)
        filings = company.get_filings(form="10-K")

        for filing in filings:
            year = filing.filing_date.year
            if year < start_yr:
                continue

            strat_path = data_dir / f"{ticker}_{year}_strategy.txt"
            risk_path = data_dir / f"{ticker}_{year}_risk.txt"
            mgmt_path = data_dir / f"{ticker}_{year}_MD&A.txt"

            if strat_path.exists() and risk_path.exists():
                continue

            tenk = filing.obj()

            strategy_text = tenk.get('Item 1')
            risk_text = tenk.get('Item 1A')
            mgmt_text = tenk.get('Item 7')

            if strategy_text:
                strat_path.write_text(strategy_text, encoding="utf-8")
            if risk_text:
                risk_path.write_text(risk_text, encoding="utf-8")
            if mgmt_text:
                mgmt_path.write_text(mgmt_text, encoding="utf-8")

            time.sleep(0.2)  # instead of 1 sec

    except Exception as e:
        print(f"Error with {ticker}: {e}")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    executor.map(process_ticker, tickers)

NameError: name 'Path' is not defined